# StreamVault Content Acquisition & Licensing Analysis
## Interactive Pipeline, Data Profiling, and Executive Modeling

This notebook provides an interactive walkthrough of the data engineering workflow designed for **Kwame Boateng, VP of Content Acquisition at StreamVault**, to advise on committing a **$14 Million licensing budget** at the October 31 board review.

### Key Objectives:
1. **Resolve Team Debate**: Evidence-based comparison between **Franchise Bundles** and **Standalone Titles**.
2. **Address Finance Skepticism**: Evaluate KPIs across both time windows (**All-Years** vs. **2000-Onward**) to address Finance VP Yaw's concerns about historical inflation and reporting standards.
3. **Full 127-Candidate Reconciliation**: Rigorously account for every row from `id_list.csv` (including blank line 69, non-numeric formats, duplicates, and 404 dead IDs).
4. **Relational Grain Integrity**: Prevent fan-out multiplication when associating cast and crew records.


In [1]:
# ==============================================================================
# BLOCK 1: Imports and Formatting Configuration
# What it does: Loads essential libraries (pandas, numpy, pathlib) and imports
# modular pipeline functions. Sets clean formatting options for executive displays.
# ==============================================================================
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Configure display formatting for clean readability
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}' if abs(x) >= 1000 else f'{x:.2f}')

# Import pipeline modular components
from pipeline.config import ID_LIST_PATH, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, TOTAL_ACQUISITION_BUDGET
from pipeline.load import load_candidate_ids, load_cached_movie_data
from pipeline.extract import TMDBExtractor
from pipeline.transform import (
    flatten_movies_data,
    engineer_financial_metrics,
    compute_kpis,
    get_franchise_rankings,
    get_standalone_rankings,
    rank_talent
)
from pipeline.validate import validate_cast_crew_grain, safe_merge_one_to_one
from pipeline.checks import profile_dataset, quality_report, build_reconciliation_table

print("Pipeline modules successfully loaded.")


Pipeline modules successfully loaded.


## Section 2: Ingestion & Candidate List Audit (127 Lines)

The candidate list (`id_list.csv`) was compiled over four years from three unverified sources: Adjoa's tracker, an inherited predecessor spreadsheet, and studio licensing emails.

Below, we audit all 127 lines, explicitly isolating:
- **Blank Row 69**: Detected and categorized as a format fault rather than a vanished record.
- **Negative Identifier `-1`** (Line 30) and **Literal Missing String `'N/A'`** (Line 111).
- **External IMDb Identifier `tt0468569`** (Line 95): Resolved to TMDb ID `155` (*The Dark Knight*), which is already present on Line 2.
- **Six Exact Duplicate IDs**: Repeated titles that would artificially inflate group totals if not removed.


In [2]:
# ==============================================================================
# BLOCK 2: Candidate ID List Audit
# What it does: Loads id_list.csv, checks every line for format or domain errors,
# and prints the breakdown across the 127 rows.
# ==============================================================================
candidate_df, valid_ids, exclusion_log = load_candidate_ids(ID_LIST_PATH)

print(f"Total lines in candidate CSV: {len(candidate_df)}")
print(f"Valid unique numeric IDs:     {len(valid_ids)}")
print(f"Total exclusions/duplicates:  {len(exclusion_log)}")

# Print breakdown of statuses across all 127 rows
print("\nCandidate Status Breakdown:")
print(candidate_df['status'].value_counts().to_string())

# Display the 10 excluded or duplicate rows
print("\nAudit Registry of All 10 Excluded Entries:")
exclusion_df = pd.DataFrame(exclusion_log)
print(exclusion_df.to_string(index=False))


Total lines in candidate CSV: 127
Valid unique numeric IDs:     117
Total exclusions/duplicates:  10

Candidate Status Breakdown:
status
VALID              117
DUPLICATE            6
EXCLUDED             3
EXTERNAL_FORMAT      1

Audit Registry of All 10 Excluded Entries:
 line    raw_id                                                   reason                  stage
   30        -1                         Negative integer identifier (-1) Door Bouncer Ingestion
   43       603              Duplicate identifier of earlier entry (603) Door Bouncer Ingestion
   60       278              Duplicate identifier of earlier entry (278) Door Bouncer Ingestion
   69   <EMPTY>                           Empty row / missing identifier Door Bouncer Ingestion
   90      1891             Duplicate identifier of earlier entry (1891) Door Bouncer Ingestion
   91       155              Duplicate identifier of earlier entry (155) Door Bouncer Ingestion
   94       550              Duplicate identifier of ea

## Section 3: Responsible Data Acquisition & Caching

Adhering to professional data engineering standards:
1. **Local Disk Caching (`data/raw/`)**: Checked before every HTTP request; re-running makes zero network calls.
2. **Session Re-use**: TCP connection kept open across calls.
3. **Explicit Timeouts**: Connect timeout (3.05s) and read timeout (10s) prevent hung background jobs.
4. **404 Dead IDs Cached**: Permanent failures are cached into `dead_ids.json` and never retried.
5. **Audit Manifest**: Every request timestamp and source is logged in `data/manifest.csv`.


In [3]:
# ==============================================================================
# BLOCK 3: Data Acquisition and Cache Verification
# What it does: Uses TMDBExtractor to load details and credits from disk cache,
# tracks dead 404 IDs, and verifies the manifest audit log.
# ==============================================================================
extractor = TMDBExtractor(raw_dir=RAW_DIR)
movies_dict, dead_ids, stats = extractor.extract_all(valid_ids)

print(f"Successfully loaded titles: {len(movies_dict)}")
print(f"Dead / 404 Identifiers:     {len(dead_ids)} -> {dead_ids}")
print(f"Acquisition Source:         {stats['cache_hits']} cache hits, {stats['network_calls']} network calls")

# Inspect manifest log
manifest_df = pd.read_csv("data/manifest.csv")
print(f"\nTotal manifest audit records: {len(manifest_df)}")
print("Recent manifest audit entries:")
print(manifest_df.tail(4)[['item_id', 'endpoint_type', 'status_code', 'source', 'timestamp_utc']].to_string(index=False))


Successfully loaded titles: 112
Dead / 404 Identifiers:     5 -> [7733901, 8471293, 8471294, 9412887, 9412888]
Acquisition Source:         229 cache hits, 0 network calls

Total manifest audit records: 6503
Recent manifest audit entries:
 item_id endpoint_type  status_code source                    timestamp_utc
  299536 movie_details       200.00  cache 2026-11-19T09:28:53.842861+00:00
  299536 movie_credits       200.00  cache 2026-11-19T09:28:54.151791+00:00
  301528 movie_details       200.00  cache 2026-11-19T09:28:54.400096+00:00
  301528 movie_credits       200.00  cache 2026-11-19T09:28:54.667517+00:00


## Section 4: Relational Grain Flattening

TMDb responses are hierarchical JSON trees. To prevent revenue duplication, we flatten the payload into three distinct tables with strict grain definitions:
1. **Films Analytical Table** (`# grain: one row per film`): Contains scalar metrics (budget, revenue, release date) and scalar collection metadata.
2. **Cast Table** (`# grain: one row per film per cast member`): Billing order, actor names, and characters.
3. **Crew Table** (`# grain: one row per film per crew member`): Directors, writers, and technical crew.


In [4]:
# ==============================================================================
# BLOCK 4: Flatten JSON into Strict Relational Tables
# What it does: Separates movie details, cast members, and crew members into three
# clean tables to enforce strict grain boundaries.
# ==============================================================================
films_raw, cast_df, crew_df = flatten_movies_data(movies_dict)

print(f"Films Table Grain (# 1 row / film):           {len(films_raw)} rows")
print(f"Cast Table Grain  (# 1 row / film / cast):    {len(cast_df)} rows")
print(f"Crew Table Grain  (# 1 row / film / crew):    {len(crew_df)} rows")

# Sample films table
print("\nSample Film Analytical Records:")
print(films_raw[['movie_id', 'title', 'release_year', 'era', 'budget', 'revenue', 'is_franchise']].head(5).to_string(index=False))


Films Table Grain (# 1 row / film):           112 rows
Cast Table Grain  (# 1 row / film / cast):    7827 rows
Crew Table Grain  (# 1 row / film / crew):    24932 rows

Sample Film Analytical Records:
 movie_id                                                  title  release_year         era         budget          revenue  is_franchise
       11                                              Star Wars          1977    Pre-2000  11,000,000.00   775,398,007.00          True
       13                                           Forrest Gump          1994    Pre-2000  55,000,000.00   677,387,716.00         False
       22 Pirates of the Caribbean: The Curse of the Black Pearl          2003 2000-Onward 140,000,000.00   655,011,224.00          True
       24                                      Kill Bill: Vol. 1          2003 2000-Onward  30,000,000.00   180,906,076.00          True
       58             Pirates of the Caribbean: Dead Man's Chest          2006 2000-Onward 200,000,000.00 1,066,17

## Section 5: Relational Grain Integrity & Fan-Out Validation

Joining cast or crew directly to the films table before financial aggregation causes severe fan-out multiplication:
- The films table has **112 rows** with **$67.2 Billion** total box office.
- The cast table has **7,827 rows** (~70 cast members per film).
- An unvalidated inner join multiplies box office to **$4.7 Trillion** (a 70x distortion).

Below, we enforce merge validation assertions (`validate='one_to_one'`, `indicator=True`) and confirm referential integrity.


In [5]:
# ==============================================================================
# BLOCK 5: Join Validation and Referential Integrity Check
# What it does: Asserts that all cast and crew records belong to verified films
# and tests safe 1-to-1 merging guards.
# ==============================================================================
grain_report = validate_cast_crew_grain(films_raw, cast_df, crew_df)

print(f"Verified Films:       {grain_report['total_films']}")
print(f"Films with Cast:      {grain_report['films_in_cast']}")
print(f"Films with Crew:      {grain_report['films_in_crew']}")
print(f"Average Cast / Film:  {grain_report['avg_cast_per_film']:.1f}")
print(f"Average Crew / Film:  {grain_report['avg_crew_per_film']:.1f}")
print(f"Unmatched Cast Film IDs: {grain_report['unmatched_cast_film_ids']}")
print(f"Unmatched Crew Film IDs: {grain_report['unmatched_crew_film_ids']}")
print("\nReferential integrity test PASSED: Zero orphaned credits detected.")


Verified Films:       112
Films with Cast:      112
Films with Crew:      112
Average Cast / Film:  69.9
Average Crew / Film:  222.6
Unmatched Cast Film IDs: []
Unmatched Crew Film IDs: []

Referential integrity test PASSED: Zero orphaned credits detected.


## Section 6: Data Profiling & Disguised Null Auditing

In entertainment databases, unknown box office gross is routinely entered as `$0` rather than `NULL`. We profile the dataset to identify disguised nulls, distribution shapes, and Tukey IQR outliers.


In [6]:
# ==============================================================================
# BLOCK 6: Exploratory Data Profiling & Outlier Detection
# What it does: Scans for $0 budgets, $0 revenues, missing dates, and calculates
# interquartile range (IQR) outlier thresholds on box office gross.
# ==============================================================================
profile = profile_dataset(films_raw)

print(f"Total Titles Analyzed:         {profile['total_records']}")
print(f"$0 Revenue Titles (Disguised): {profile['zero_revenue_count']} (0.0% of verified catalog)")
print(f"$0 Budget Titles  (Disguised): {profile['zero_budget_count']} (0.0% of verified catalog)")
print(f"Missing Release Dates:         {profile['missing_release_dates']}")
print(f"Era Distribution:              {profile['era_breakdown']}")
print(f"Franchise Distribution:        {profile['franchise_breakdown']}")

rev_stats = profile['revenue_stats']
fence = rev_stats['p75'] + 1.5 * rev_stats['iqr']
print(f"\nRevenue Distribution (Non-Zero):")
print(f"  25th Percentile: ${rev_stats['p25']/1e6:.1f}M")
print(f"  Median:          ${rev_stats['median']/1e6:.1f}M")
print(f"  Mean:            ${rev_stats['mean']/1e6:.1f}M")
print(f"  75th Percentile: ${rev_stats['p75']/1e6:.1f}M")
print(f"  IQR Outlier Fence (Q3 + 1.5*IQR): ${fence/1e6:.1f}M")
print(f"  Identified Outliers (> ${fence/1e6:.1f}M): {profile['revenue_outlier_count']} titles")


Total Titles Analyzed:         112
$0 Revenue Titles (Disguised): 0 (0.0% of verified catalog)
$0 Budget Titles  (Disguised): 0 (0.0% of verified catalog)
Missing Release Dates:         0
Era Distribution:              {'2000-Onward': 71, 'Pre-2000': 41}
Franchise Distribution:        {'Franchise': 96, 'Standalone': 16}

Revenue Distribution (Non-Zero):
  25th Percentile: $295.3M
  Median:          $519.6M
  Mean:            $599.7M
  75th Percentile: $872.7M
  IQR Outlier Fence (Q3 + 1.5*IQR): $1738.7M
  Identified Outliers (> $1738.7M): 2 titles


## Section 7: Feature Engineering & Parameterized Financial Rules

We compute:
- **Gross Profit**: `revenue - budget`
- **Return on Cost (ROI Multiple)**: `revenue / budget`
- **Commercial Tiers**:
  - `Blockbuster Hit`: Return >= 2.5x (theatrical/commercial breakeven rule)
  - `Moderate Return`: 1.0x <= Return < 2.5x
  - `Flop / Loss`: Return < 1.0x
  - `Unknown`: Disguised null ($0 revenue or $0 budget)

We parameterize this calculation so we can test both baseline (treat $0 as missing) and sensitivity (treat $0 as flop) rules.


In [7]:
# ==============================================================================
# BLOCK 7: Vectorized Feature Engineering
# What it does: Calculates profit, ROI multiples, and commercial tier categories
# using fast, vectorized NumPy operations without slow row loops.
# ==============================================================================
# Baseline: treat $0 revenue as missing data (unknown theatrical gross)
films_baseline = engineer_financial_metrics(films_raw, treat_zero_revenue_as_missing=True)

# Sensitivity: treat $0 revenue as genuine 0x commercial flop
films_sensitivity = engineer_financial_metrics(films_raw, treat_zero_revenue_as_missing=False)

print("Commercial Outcome Distribution (Baseline):")
print(films_baseline['commercial_tier'].value_counts().to_string())

print("\nSample Cleaned Financial Records:")
print(films_baseline[['title', 'release_year', 'budget', 'revenue', 'return_on_cost', 'commercial_tier']].head(5).to_string(index=False))


Commercial Outcome Distribution (Baseline):
commercial_tier
Blockbuster Hit    103
Moderate Return      9

Sample Cleaned Financial Records:
                                                 title  release_year         budget          revenue  return_on_cost commercial_tier
                                             Star Wars          1977  11,000,000.00   775,398,007.00           70.49 Blockbuster Hit
                                          Forrest Gump          1994  55,000,000.00   677,387,716.00           12.32 Blockbuster Hit
Pirates of the Caribbean: The Curse of the Black Pearl          2003 140,000,000.00   655,011,224.00            4.68 Blockbuster Hit
                                     Kill Bill: Vol. 1          2003  30,000,000.00   180,906,076.00            6.03 Blockbuster Hit
            Pirates of the Caribbean: Dead Man's Chest          2006 200,000,000.00 1,066,179,747.00            5.33 Blockbuster Hit


## Section 8: Dual-Window KPI Analysis: Defusing Finance Skepticism

Yaw in Finance distrusts pre-2000 numbers due to historical inflation and differing reporting standards.
To provide indisputable evidence, we compute KPIs across **both** time windows:
1. **All-Years Window**: Complete 112-title catalog.
2. **2000-Onward Window (Finance View)**: Modern releases only.


In [8]:
# ==============================================================================
# BLOCK 8: Comparative Dual-Window KPI Evaluation
# What it does: Computes median ROI, total gross, and hit rates for franchises vs
# standalones across both eras, proving that the recommendation holds in both.
# ==============================================================================
kpis_all_years = compute_kpis(films_baseline, era_filter=None)
kpis_post_2000 = compute_kpis(films_baseline, era_filter="2000-Onward")

# Build executive comparison table
comp_rows = [
    {
        "Era Window": "All-Years (Full Catalog)",
        "Group": "Franchise Bundles",
        "Titles": kpis_all_years['Franchise']['total_titles'],
        "Median ROI": f"{kpis_all_years['Franchise']['median_roi']:.2f}x",
        "Mean ROI": f"{kpis_all_years['Franchise']['mean_roi']:.2f}x",
        "Total Revenue": f"${kpis_all_years['Franchise']['total_revenue']/1e9:.2f}B",
        "Hit Rate (>=2.5x)": f"{kpis_all_years['Franchise']['hit_rate']*100:.1f}%",
        "Failure Rate (<1.0x)": f"{kpis_all_years['Franchise']['failure_rate']*100:.1f}%",
        "Mean Rating": f"{kpis_all_years['Franchise']['mean_vote_average']:.2f}/10"
    },
    {
        "Era Window": "All-Years (Full Catalog)",
        "Group": "Standalone Titles",
        "Titles": kpis_all_years['Standalone']['total_titles'],
        "Median ROI": f"{kpis_all_years['Standalone']['median_roi']:.2f}x",
        "Mean ROI": f"{kpis_all_years['Standalone']['mean_roi']:.2f}x",
        "Total Revenue": f"${kpis_all_years['Standalone']['total_revenue']/1e9:.2f}B",
        "Hit Rate (>=2.5x)": f"{kpis_all_years['Standalone']['hit_rate']*100:.1f}%",
        "Failure Rate (<1.0x)": f"{kpis_all_years['Standalone']['failure_rate']*100:.1f}%",
        "Mean Rating": f"{kpis_all_years['Standalone']['mean_vote_average']:.2f}/10"
    },
    {
        "Era Window": "2000-Onward (Finance View)",
        "Group": "Franchise Bundles",
        "Titles": kpis_post_2000['Franchise']['total_titles'],
        "Median ROI": f"{kpis_post_2000['Franchise']['median_roi']:.2f}x",
        "Mean ROI": f"{kpis_post_2000['Franchise']['mean_roi']:.2f}x",
        "Total Revenue": f"${kpis_post_2000['Franchise']['total_revenue']/1e9:.2f}B",
        "Hit Rate (>=2.5x)": f"{kpis_post_2000['Franchise']['hit_rate']*100:.1f}%",
        "Failure Rate (<1.0x)": f"{kpis_post_2000['Franchise']['failure_rate']*100:.1f}%",
        "Mean Rating": f"{kpis_post_2000['Franchise']['mean_vote_average']:.2f}/10"
    },
    {
        "Era Window": "2000-Onward (Finance View)",
        "Group": "Standalone Titles",
        "Titles": kpis_post_2000['Standalone']['total_titles'],
        "Median ROI": f"{kpis_post_2000['Standalone']['median_roi']:.2f}x",
        "Mean ROI": f"{kpis_post_2000['Standalone']['mean_roi']:.2f}x",
        "Total Revenue": f"${kpis_post_2000['Standalone']['total_revenue']/1e9:.2f}B",
        "Hit Rate (>=2.5x)": f"{kpis_post_2000['Standalone']['hit_rate']*100:.1f}%",
        "Failure Rate (<1.0x)": f"{kpis_post_2000['Standalone']['failure_rate']*100:.1f}%",
        "Mean Rating": f"{kpis_post_2000['Standalone']['mean_vote_average']:.2f}/10"
    }
]

comp_df = pd.DataFrame(comp_rows)
print("EXECUTIVE DUAL-WINDOW COMPARISON TABLE:")
print(comp_df.to_string(index=False))


EXECUTIVE DUAL-WINDOW COMPARISON TABLE:
                Era Window             Group  Titles Median ROI Mean ROI Total Revenue Hit Rate (>=2.5x) Failure Rate (<1.0x) Mean Rating
  All-Years (Full Catalog) Franchise Bundles      96      5.62x    8.54x       $62.84B             94.8%                 0.0%     7.46/10
  All-Years (Full Catalog) Standalone Titles      16      3.84x    8.28x        $4.32B             75.0%                 0.0%     8.37/10
2000-Onward (Finance View) Franchise Bundles      63      5.05x    5.41x       $49.96B             95.2%                 0.0%     7.33/10
2000-Onward (Finance View) Standalone Titles       8      3.84x    4.90x        $2.83B             87.5%                 0.0%     8.26/10


## Section 9: Executive Acquisition Shortlists & Talent Rankings

To support Kwame's negotiation team at the October 31 review, we generate:
1. **Franchise Collection Shortlist**: Ranked by cumulative box office and ROI.
2. **Standalone Cherry-Pick Shortlist**: Ranked by capital efficiency.
3. **Talent Attribution**: Top directors and lead actors by commercial track record.


In [9]:
# ==============================================================================
# BLOCK 9: Generate Actionable Acquisition Shortlists
# What it does: Ranks top franchise bundles, top standalone titles, and evaluates
# directors and actors without distorting the underlying film grain.
# ==============================================================================
franchise_ranks = get_franchise_rankings(films_baseline)
standalone_ranks = get_standalone_rankings(films_baseline)
dir_summary, act_summary = rank_talent(films_baseline, cast_df, crew_df)

print("TOP 5 FRANCHISE BUNDLE TARGETS:")
print(franchise_ranks[['collection_name', 'film_count', 'total_revenue', 'median_roi', 'mean_vote']].head(5).to_string(index=False))

print("\nTOP 5 STANDALONE CHERRY-PICK TARGETS (Post-2000):")
post_standalones = standalone_ranks[standalone_ranks['release_year'] >= 2000]
print(post_standalones[['title', 'release_year', 'return_on_cost', 'revenue', 'vote_average']].head(5).to_string(index=False))

print("\nTOP 5 DIRECTORS BY COMMERCIAL PERFORMANCE:")
print(dir_summary[['crew_name', 'film_count', 'total_revenue', 'median_roi']].head(5).to_string(index=False))


TOP 5 FRANCHISE BUNDLE TARGETS:
                    collection_name  film_count    total_revenue  median_roi  mean_vote
            The Avengers Collection           4 7,919,284,026.00        6.87       7.95
            Harry Potter Collection           8 7,759,777,709.00        6.17       7.83
Pirates of the Caribbean Collection           5 4,525,525,744.00        3.46       7.15
               Star Wars Collection           6 4,432,411,744.00       13.40       7.53
               Toy Story Collection           4 3,039,690,868.00        5.83       7.72

TOP 5 STANDALONE CHERRY-PICK TARGETS (Post-2000):
        title  release_year  return_on_cost        revenue  vote_average
Spirited Away          2001           14.47 274,925,095.00          8.54
    Inception          2010            5.24 839,030,630.00          8.37
 Interstellar          2014            4.52 746,606,706.00          8.50
      Memento          2000            4.45  40,060,108.00          8.18
 The Departed          2

## Section 10: 6-Stage Candidate Reconciliation Table

The audit ledger below accounts for every single candidate row from `id_list.csv` (127 lines) down to the final verified catalog (112 titles).


In [10]:
# ==============================================================================
# BLOCK 10: 6-Stage Candidate Reconciliation Ledger
# What it does: Builds and displays the complete reconciliation table proving
# 100% mathematical accountability for all 127 raw candidate lines.
# ==============================================================================
reconciliation = build_reconciliation_table(candidate_df, dead_ids, films_baseline)
recon_df = pd.DataFrame(reconciliation)

print("6-STAGE CANDIDATE RECONCILIATION LEDGER:")
print(recon_df.to_string(index=False))


6-STAGE CANDIDATE RECONCILIATION LEDGER:
                                   Stage  Rows  Delta                                                                                                      Reason
              1. Raw Candidate CSV Input   127      0                                    Initial candidate list from tracker, legacy spreadsheet, and studio desk
          2. Ingestion Format Validation   124     -3                                      Dynamically removed blank rows, negative IDs, and literal 'N/A' string
    3. External Identifier Harmonization   123     -1                                  IMDb ID tt0468569 resolves to TMDb 155 (already present in candidate list)
               4. Exact ID Deduplication   117     -6                                                   Deduplicated repeated IDs (155, 278, 550, 603, 680, 1891)
           5. Live API Resolution (404s)   112     -5 TMDb API returned 404 Not Found for 5 invalid/deprecated IDs: [7733901, 8471293, 8471294, 94128